# Benchmark OCR — modelos que exigem GPU

Roda no Google Colab os modelos que não cabem no notebook do trabalho (sem GPU, 7,5 GB de RAM).

**Antes de começar**
1. Suba o arquivo `tcc_colab.zip` para a raiz do seu Google Drive (*Meu Drive*).
2. Aqui no Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.
3. Rode as células em ordem (**Ambiente de execução → Executar tudo**).

Os resultados vão para `Meu Drive/tcc_resultados_gpu/`. Se a sessão cair, é só rodar tudo de novo: páginas já salvas são puladas.

In [ ]:
# 1. Confere a GPU (deve aparecer "Tesla T4")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Conecta o Google Drive e descompacta o projeto
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/tcc && mkdir -p /content/tcc
!unzip -q /content/drive/MyDrive/tcc_colab.zip -d /content/tcc
%cd /content/tcc
!ls data/corpus/images

In [ ]:
# 3. Dependências (torch e transformers já vêm no Colab)
!pip install -q ollama psutil python-dotenv gdown pytesseract
# sentencepiece/protobuf: tokenizador do TrOCR
!pip install -q sentencepiece protobuf
# --no-deps: o craft-text-detector fixa versões antigas do OpenCV
!pip install -q --no-deps craft-text-detector

In [ ]:
# 4. Instala e inicia o Ollama em segundo plano
# O instalador do Ollama precisa do zstd, que não vem no Colab
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version'); print('Ollama no ar'); break
    except Exception:
        time.sleep(2)

In [ ]:
# 5. Baixa os modelos (leva alguns minutos)
for tag in ['qwen2.5vl:7b', 'gemma4:e4b', 'deepseek-ocr:3b', 'MedAIBase/PaddleOCR-VL:0.9b']:
    print('==', tag)
    !ollama pull {tag} 2>&1 | tail -1

In [ ]:
# 6. Extração. Modelos rodam um de cada vez; cada página é salva ao terminar.
MODELOS = 'trocr-base trocr-large paddleocr-vl deepseek-ocr gemma4-e4b qwen-7b'
SAIDA = '/content/drive/MyDrive/tcc_resultados_gpu'
!python benchmark/extrair.py --models {MODELOS} --images data/corpus/images --saida {SAIDA}

In [ ]:
# 7. Situação final
!cat /content/drive/MyDrive/tcc_resultados_gpu/_estado/resumo.txt

**Depois:** baixe a pasta `tcc_resultados_gpu` do Drive e coloque no projeto como `data/results/extracao_gpu/`.

Os tempos medidos aqui são de GPU e **não são comparáveis** com os do notebook do trabalho; servem só para a tabela do ambiente de nuvem.